## Import Libraries

In [114]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
import json

## Load Data

In [99]:
# Load dataset
DATA_DIR = Path("./data")

In [100]:
# Check classes
class_dirs = sorted([
    p for p in DATA_DIR.iterdir()
    if p.is_dir()
])

print("Number of class folders:", len(class_dirs))
print([p.name for p in class_dirs[:10]])

Number of class folders: 38
['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight']


In [101]:
# Create a dataframe that has paths, class names and species
records = []

for class_dir in class_dirs:
    class_name = class_dir.name
    species = class_name.split("___")[0]

    for image_path in class_dir.iterdir():
        if image_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
            records.append({
                "path": str(image_path),
                "class_name": class_name,
                "species": species
            })

df = pd.DataFrame(records)

df.head()

,path,class_name,species
0,data/Apple___Apple_scab/14c623e5-051c-42f6-9e4...,Apple___Apple_scab,Apple
1,data/Apple___Apple_scab/bceb695a-e877-4b2b-bf0...,Apple___Apple_scab,Apple
2,data/Apple___Apple_scab/1f6abf22-93fa-48f0-a50...,Apple___Apple_scab,Apple
3,data/Apple___Apple_scab/623fb084-dbac-42fb-b63...,Apple___Apple_scab,Apple
4,data/Apple___Apple_scab/be517ff5-5e63-4f6a-9f5...,Apple___Apple_scab,Apple


In [102]:
print(f"Images: {len(df):,}")
print(f"Classes: {df['class_name'].nunique()}")
print(f"Species: {df['species'].nunique()}")

Images: 54,305
Classes: 38
Species: 14


## Select OOD classes

In [ ]:
# Get summary based on species
species_summary = (
    df.groupby("species")
      .agg(
          n_images=("path", "size"),
          n_classes=("class_name", "nunique")
      )
      .sort_values("n_images", ascending=False)
      .reset_index()
)

species_summary

,species,n_images,n_classes
0,Tomato,18160,10
1,Orange,5507,1
2,Soybean,5090,1
3,Grape,4062,4
4,Corn_(maize),3852,4
5,Apple,3171,4
6,Peach,2657,2
7,"Pepper,_bell",2475,2
8,Potato,2152,3
9,Cherry_(including_sour),1906,2


Conclusion:
- Grape and corn were selected as the held-out crop species. Both contain approximately 4,000 images distributed across four health/disease classes, providing sufficiently large and heterogeneous OOD evaluation sets while avoiding removal of the disproportionately represented tomato classes.

## Condition Pre-existing leaf-grouping Metadata

The original PlantVillage repository includes leaf-grouping metadata specifically for this reason. 

(assignment explicitly warns that splits need to respect the underlying data structure rather than simply randomising individual records.)

In [110]:
# View split info from leaf metadata file
train_split_file = DATA_DIR / "color_train.txt"
test_split_file = DATA_DIR / "color_test.txt"

with open(train_split_file) as f:
    train_lines = [line.strip() for line in f if line.strip()]

with open(test_split_file) as f:
    test_lines = [line.strip() for line in f if line.strip()]

print(train_lines[:3])
print(test_lines[:3])

['data/Raspberry___healthy/a37f34f7-022d-461a-8a3d-95f5cd774e35___Mary_HL 9155.JPG', 'data/Raspberry___healthy/368f3c83-5bd2-4906-b418-57b6ceff9bfe___Mary_HL 9160.JPG', 'data/Raspberry___healthy/98226f66-15a1-40d5-9b62-04ae5b4fd8d5___Mary_HL 9159.JPG']
['data/Raspberry___healthy/6c2e049f-38c9-42c5-864f-613e6e12d59e___Mary_HL 6318.JPG', 'data/Raspberry___healthy/8e8297fd-5e23-44f2-be98-92f7b8908f81___Mary_HL 6315.JPG', 'data/Raspberry___healthy/67cff824-a58b-40c3-93a2-7dead429b1f0___Mary_HL 6322.JPG']


In [ ]:
# Add column to df to indicate split category
official_train = set(train_lines)
official_test = set(test_lines)

def assign_official_split(path):
    if path in official_train:
        return "train"
    elif path in official_test:
        return "test"
    return "unmatched"

df["official_split"] = df["path"].map(assign_official_split)

In [85]:
df.head()

,path,class_name,species,official_split
0,data/Apple___Apple_scab/14c623e5-051c-42f6-9e4...,Apple___Apple_scab,Apple,train
1,data/Apple___Apple_scab/bceb695a-e877-4b2b-bf0...,Apple___Apple_scab,Apple,train
2,data/Apple___Apple_scab/1f6abf22-93fa-48f0-a50...,Apple___Apple_scab,Apple,train
3,data/Apple___Apple_scab/623fb084-dbac-42fb-b63...,Apple___Apple_scab,Apple,test
4,data/Apple___Apple_scab/be517ff5-5e63-4f6a-9f5...,Apple___Apple_scab,Apple,train


## Filter Train and Test for our OOD species

In [109]:
# Separate in-distribution and out-of-distribution datasets
OOD_SPECIES = [
    "Grape",
    "Corn_(maize)"
]

ood_df = df[df["species"].isin(OOD_SPECIES)].copy()
id_df = df[~df["species"].isin(OOD_SPECIES)].copy()

In [ ]:
# Create train and test split based on in-distribution dataset
train_val_id = id_df[
    id_df["official_split"] == "train"
].copy()

test_id = id_df[
    id_df["official_split"] == "test"
].copy()

In [108]:
# Further separate out-of-distribution dataset into specific datasets for grape and corn respectively
ood_grape = ood_df[
    ood_df["species"] == "Grape"
].copy()

ood_corn = ood_df[
    ood_df["species"] == "Corn_(maize)"
].copy()

## Create Validation Set

In [ ]:
# Use stratified sampling to create a train, validation split based on the initial splitted training data (10% val)
train_id, val_id = train_test_split(
    train_val_id,
    test_size=0.10,
    random_state=42,
    stratify=train_val_id["class_name"]
)

## Final Integrity Checks

In [106]:
# Expected number of ID classes
assert train_id["class_name"].nunique() == 30
assert val_id["class_name"].nunique() == 30
assert test_id["class_name"].nunique() == 30

# OOD species must be fully excluded from all ID splits
OOD_SPECIES = {"Grape", "Corn_(maize)"}

assert OOD_SPECIES.isdisjoint(set(train_id["species"]))
assert OOD_SPECIES.isdisjoint(set(val_id["species"]))
assert OOD_SPECIES.isdisjoint(set(test_id["species"]))

# OOD datasets should contain only the intended species
assert set(ood_grape["species"].unique()) == {"Grape"}
assert set(ood_corn["species"].unique()) == {"Corn_(maize)"}

# No image should occur in more than one ID split
assert set(train_id["path"]).isdisjoint(set(val_id["path"]))
assert set(train_id["path"]).isdisjoint(set(test_id["path"]))
assert set(val_id["path"]).isdisjoint(set(test_id["path"]))

print("All split integrity checks passed.")

All split integrity checks passed.


## Summary Statistics

In [107]:
split_summary = pd.DataFrame({
    "split": [
        "Train ID",
        "Validation ID",
        "Test ID",
        "OOD Grape",
        "OOD Corn"
    ],
    "n_images": [
        len(train_id),
        len(val_id),
        len(test_id),
        len(ood_grape),
        len(ood_corn)
    ],
    "n_classes": [
        train_id["class_name"].nunique(),
        val_id["class_name"].nunique(),
        test_id["class_name"].nunique(),
        ood_grape["class_name"].nunique(),
        ood_corn["class_name"].nunique()
    ],
    "n_species": [
        train_id["species"].nunique(),
        val_id["species"].nunique(),
        test_id["species"].nunique(),
        ood_grape["species"].nunique(),
        ood_corn["species"].nunique()
    ]
})

split_summary

,split,n_images,n_classes,n_species
0,Train ID,33508,30,12
1,Validation ID,3724,30,12
2,Test ID,9159,30,12
3,OOD Grape,4062,4,1
4,OOD Corn,3852,4,1


In [111]:
class_distribution = pd.concat(
    [
        train_id["class_name"].value_counts(normalize=True).rename("train"),
        val_id["class_name"].value_counts(normalize=True).rename("val"),
        test_id["class_name"].value_counts(normalize=True).rename("test"),
    ],
    axis=1
).fillna(0)

class_distribution.head()

,train,val,test
class_name,,,
Orange___Haunglongbing_(Citrus_greening),0.121583,0.121643,0.106999
Tomato___Tomato_Yellow_Leaf_Curl_Virus,0.114331,0.114393,0.120100
Soybean___healthy,0.110571,0.110634,0.106234
Peach___Bacterial_spot,0.047123,0.047261,0.059177
Tomato___Bacterial_spot,0.045422,0.045381,0.047603


## Save the split information

In [113]:
SPLIT_DIR = Path("./splits")
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

train_id.to_csv(SPLIT_DIR / "train_id.csv", index=False)
val_id.to_csv(SPLIT_DIR / "val_id.csv", index=False)
test_id.to_csv(SPLIT_DIR / "test_id.csv", index=False)
ood_grape.to_csv(SPLIT_DIR / "ood_grape.csv", index=False)
ood_corn.to_csv(SPLIT_DIR / "ood_corn.csv", index=False)

In [ ]:
# Map classes to indices
classes = sorted(train_id["class_name"].unique())

class_to_idx = {
    class_name: idx
    for idx, class_name in enumerate(classes)
}

with open(SPLIT_DIR / "class_to_idx.json", "w") as f:
    json.dump(class_to_idx, f, indent=2)

In [117]:
split_config = {
    "seed": 42,
    "ood_species": [
        "Grape",
        "Corn_(maize)"
    ],
    "n_original_images": 54305,
    "n_original_classes": 38,
    "n_original_species": 14,
    "n_id_classes": 30,
    "validation_strategy": (
        "10% stratified split from the official PlantVillage "
        "training partition using class_name."
    ),
    "test_strategy": (
        "Official PlantVillage test partition retained unchanged "
        "for known-species ID evaluation."
    ),
    "ood_strategy": (
        "All Grape and Corn_(maize) images removed from ID training, "
        "validation and testing and reserved for OOD evaluation."
    )
}

with open(SPLIT_DIR / "split_config.json", "w") as f:
    json.dump(split_config, f, indent=2)

## Final Outcome

In [119]:
print("Dataset setup complete")
print("-" * 40)

for _, row in split_summary.iterrows():
    print(
        f"{row['split']:<15} "
        f"{row['n_images']:>6,} images | "
        f"{row['n_classes']:>2} classes | "
        f"{row['n_species']:>2} species"
    )

print("-" * 40)
print(f"ID classes: {len(class_to_idx)}")
print("Held-out OOD species: Grape, Corn_(maize)")

Dataset setup complete
----------------------------------------
Train ID        33,508 images | 30 classes | 12 species
Validation ID    3,724 images | 30 classes | 12 species
Test ID          9,159 images | 30 classes | 12 species
OOD Grape        4,062 images |  4 classes |  1 species
OOD Corn         3,852 images |  4 classes |  1 species
----------------------------------------
ID classes: 30
Held-out OOD species: Grape, Corn_(maize)
